# [Notebook]Unihack26_Cross_Source_Audit
Canonical fingerprints for Parcel3D and official VisA, CPU only. Checks pinned image identities, exact duplicates and bounded perceptual candidates. Data stays on Kaggle. Test pixels are used only for hygiene, never model tuning. No split reassignment, training or automatic promotion. Parcel3D original renders academic-only; VisA CC BY 4.0.

In [ ]:
from pathlib import Path
import sys,json,hashlib
SOURCES={'ml/__init__.py': '', 'ml/data/__init__.py': '', 'ml/data/parcel3d_audit.py': '"""CPU-only audit of a mounted Parcel3D input. Does not copy its image archive."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport platform\nfrom collections import Counter, deque\nfrom datetime import UTC, datetime\nfrom pathlib import Path, PurePosixPath\n\nfrom PIL import Image\n\nSOURCE = "ayesha19765/parcel3d"\nORIGINAL = "https://zenodo.org/records/8032204"\nALLOWED_SPLITS = ("train", "validation", "test")\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef within_root(root: Path, relative: str) -> Path:\n    relative = relative.replace("\\\\", "/")\n    parsed = PurePosixPath(relative)\n    if parsed.is_absolute() or ".." in parsed.parts or ":" in relative:\n        raise ValueError("UNSAFE_ANNOTATION_PATH")\n    result = (root / Path(*parsed.parts)).resolve()\n    if not result.is_relative_to(root.resolve()):\n        raise ValueError("PATH_OUTSIDE_INPUT")\n    return result\n\n\ndef find_annotations(input_root: Path) -> dict[str, Path]:\n    # Kaggle can expose both legacy and owner-qualified aliases of one input.\n    # Annotation files sit above per-image directories; never walk 13,400 samples\n    # just to locate three files. Bound discovery and compare duplicate contents.\n    roots, queue, visited = [], deque([(input_root, 0)]), set()\n    while queue:\n        directory, depth = queue.popleft()\n        resolved = directory.resolve()\n        if resolved in visited:\n            continue\n        visited.add(resolved)\n        if len(visited) > 200:\n            raise RuntimeError("ANNOTATION_DISCOVERY_LIMIT")\n        if all((directory / f"parcel3d_{split}.json").is_file() for split in ALLOWED_SPLITS):\n            roots.append(directory)\n            continue\n        if depth < 5:\n            for child in sorted(directory.iterdir()):\n                if (\n                    child.name not in {*ALLOWED_SPLITS, "misc"}\n                    and not child.name.startswith(".")\n                    and child.is_dir()\n                ):\n                    queue.append((child, depth + 1))\n    if not roots:\n        raise RuntimeError("ANNOTATION_ROOT_NOT_FOUND")\n    roots.sort(key=lambda path: (len(path.parts), str(path)))\n    if len(roots) > 1:\n        digests = {\n            tuple(sha256_file(root / f"parcel3d_{split}.json") for split in ALLOWED_SPLITS)\n            for root in roots\n        }\n        if len(digests) != 1:\n            raise RuntimeError("ANNOTATION_ALIASES_HAVE_DIFFERENT_CONTENT")\n    print(f"Annotation roots: {len(roots)} equivalent mount(s); selected {roots[0]}", flush=True)\n    return {split: roots[0] / f"parcel3d_{split}.json" for split in ALLOWED_SPLITS}\n\n\ndef resolve_image(annotation_path: Path, image_record: dict, split: str) -> Path | None:\n    filename = image_record.get("file_name")\n    if not isinstance(filename, str):\n        return None\n    candidates = [\n        within_root(annotation_path.parent, filename),\n        within_root(annotation_path.parent, f"{split}/{filename}"),\n    ]\n    for candidate in candidates:\n        if candidate.is_file():\n            return candidate\n    return None\n\n\ndef audit(input_root: Path, output_root: Path, source_version: int = 1) -> dict:\n    input_root = input_root.resolve()\n    output_root = output_root.resolve()\n    if output_root.is_relative_to(input_root):\n        raise ValueError("OUTPUT_MUST_NOT_BE_INSIDE_READ_ONLY_INPUT")\n    output_root.mkdir(parents=True, exist_ok=True)\n    print("Locating Parcel3D annotation roots (bounded discovery)", flush=True)\n    annotations = find_annotations(input_root)\n    result = {\n        "schema_version": "parcel3d-audit-v2",\n        "source": SOURCE,\n        "source_version": source_version,\n        "original_record": ORIGINAL,\n        "checked_at": datetime.now(UTC).isoformat(),\n        "runtime": {"python": platform.python_version(), "platform": platform.platform()},\n        "source_kind": "SYNTHETIC",\n        "splits": {},\n        "errors": [],\n        "license": {\n            "mirror_metadata": "Apache-2.0; not authoritative for upstream assets",\n            "original_full_dataset": "Academic research only; Zenodo other-nc",\n            "textureless_obj_meshes": "CC-BY-4.0, separate grant in original record",\n            "cardboard_textures": "CC-BY-4.0, separate record 8041823",\n            "training_policy": "Full rendered-image training remains gated pending scope review.",\n            "commercial_release": "NOT_CLEARED_FOR_ORIGINAL_RENDERED_IMAGES",\n            "reference_matching_labels": "Must inspect; COCO damage labels alone do not establish cross-view correspondence or evidence sufficiency.",\n        },\n    }\n    for package in ("numpy", "Pillow", "torch", "opencv-python"):\n        try:\n            result["runtime"][package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            pass\n    group_sets = {}\n    sample_directories = set()\n    for split, path in annotations.items():\n        print(f"Auditing {split}: {path.name}", flush=True)\n        document = json.loads(path.read_text(encoding="utf-8"))\n        images = document.get("images", [])\n        labels = document.get("annotations", [])\n        record = {\n            "annotation_path": str(path.relative_to(input_root)),\n            "annotation_sha256": sha256_file(path),\n            "annotation_bytes": path.stat().st_size,\n            "top_level_keys": sorted(document),\n            "image_count": len(images),\n            "annotation_count": len(labels),\n            "categories": document.get("categories", []),\n            "image_fields": dict(Counter(k for image in images for k in image)),\n            "annotation_fields": dict(Counter(k for annotation in labels for k in annotation)),\n            "sample_records": images[:2],\n            "sample_annotations": labels[:2],\n            "samples": [],\n        }\n        ids = [image.get("id") for image in images]\n        record["duplicate_image_ids"] = len(ids) - len(set(ids))\n        group_key = next(\n            (\n                key\n                for key in ("mesh_id", "object_id", "parcel_id", "model_id")\n                if images and all(key in item for item in images)\n            ),\n            None,\n        )\n        record["explicit_group_key"] = group_key\n        record["group_split_verified"] = False\n        if group_key:\n            group_sets[split] = {str(item[group_key]) for item in images}\n        # Deterministic stratified positions; no exhaustive image read or copy.\n        positions = sorted({int(i * max(len(images) - 1, 0) / 7) for i in range(8)})\n        for index in positions if images else []:\n            image_record = images[index]\n            image_path = resolve_image(path, image_record, split)\n            sample = {\n                "image_id": image_record.get("id"),\n                "file_name": image_record.get("file_name"),\n            }\n            if image_path is None:\n                sample["status"] = "PATH_UNRESOLVED"\n                result["errors"].append(\n                    {"split": split, "image_id": sample["image_id"], "code": "PATH_UNRESOLVED"}\n                )\n            else:\n                try:\n                    with Image.open(image_path) as image:\n                        image.load()\n                        sample.update(\n                            status="READ_OK",\n                            width=image.width,\n                            height=image.height,\n                            format=image.format,\n                            mode=image.mode,\n                        )\n                    sample["sha256"] = sha256_file(image_path)\n                    sample["bytes"] = image_path.stat().st_size\n                    sample["relative_path"] = str(image_path.relative_to(input_root))\n                    sample_directories.add(image_path.parent)\n                except (OSError, ValueError) as exc:\n                    sample.update(status="READ_FAILED", error_type=type(exc).__name__)\n                    result["errors"].append(\n                        {"split": split, "image_id": sample["image_id"], "code": "READ_FAILED"}\n                    )\n            record["samples"].append(sample)\n        result["splits"][split] = record\n    root = annotations["train"].parent\n    extensions = Counter()\n    mesh_examples = []\n    metadata_examples = []\n    # Inventory only sampled image neighborhoods. Do not imply a complete count.\n    print(f"Inspecting metadata/meshes near {len(sample_directories)} sampled images", flush=True)\n    for sample_directory in sorted(sample_directories):\n        for directory, _, filenames in os.walk(sample_directory):\n            for filename in sorted(filenames):\n                suffix = Path(filename).suffix.lower()\n                extensions[suffix] += 1\n                path = Path(directory) / filename\n                if suffix == ".obj" and len(mesh_examples) < 8:\n                    mesh_examples.append(\n                        {\n                            "path": str(path.relative_to(root)),\n                            "bytes": path.stat().st_size,\n                            "sha256": sha256_file(path),\n                        }\n                    )\n                if (\n                    suffix == ".json"\n                    and path not in annotations.values()\n                    and len(metadata_examples) < 8\n                ):\n                    meta = {"path": str(path.relative_to(root)), "bytes": path.stat().st_size}\n                    if path.stat().st_size < 100_000:\n                        try:\n                            meta["document"] = json.loads(path.read_text(encoding="utf-8"))\n                        except (ValueError, UnicodeError):\n                            meta["status"] = "UNREADABLE_JSON"\n                    metadata_examples.append(meta)\n    result["inventory"] = {\n        "scope": "SAMPLED_IMAGE_NEIGHBORHOODS",\n        "sample_directory_count": len(sample_directories),\n        "sampled_extension_counts": dict(extensions),\n        "mesh_examples": mesh_examples,\n        "metadata_examples": metadata_examples,\n    }\n    if len(group_sets) == 3:\n        overlaps = {\n            f"{a}:{b}": len(group_sets[a] & group_sets[b])\n            for a, b in (("train", "validation"), ("train", "test"), ("validation", "test"))\n        }\n        result["group_overlap_counts"] = overlaps\n        for record in result["splits"].values():\n            record["group_split_verified"] = not any(overlaps.values())\n    result["mount_read_gate"] = (\n        "PASS"\n        if not result["errors"]\n        and all(\n            any(s["status"] == "READ_OK" for s in record["samples"])\n            for record in result["splits"].values()\n        )\n        else "FAIL"\n    )\n    result["correspondence_label_gate"] = "PENDING_REVIEW"\n    result["commercial_license_gate"] = "HOLD_ORIGINAL_RENDERS"\n    result["full_archive_copied"] = False\n    target = output_root / "parcel3d-audit.json"\n    target.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")\n    (output_root / "audit-sha256.txt").write_text(\n        sha256_file(target) + "  parcel3d-audit.json\\n", encoding="ascii"\n    )\n    print(\n        json.dumps(\n            {\n                "mount_read_gate": result["mount_read_gate"],\n                "counts": {k: v["image_count"] for k, v in result["splits"].items()},\n                "outputs": str(output_root),\n            },\n            ensure_ascii=False,\n        )\n    )\n    return result\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--input", type=Path, default=Path("/kaggle/input"))\n    parser.add_argument("--output", type=Path, default=Path("/kaggle/working/unihack26-audit"))\n    parser.add_argument("--source-version", type=int, default=1)\n    args = parser.parse_args()\n    audit(args.input, args.output, args.source_version)\n', 'ml/data/visa_audit.py': '"""Audit official VisA on Kaggle; export metadata only, never the image archive."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport json\nimport tarfile\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\nfrom urllib.request import urlopen\n\nimport numpy as np\nfrom PIL import Image\n\nREVISION = "2a692ab575001cbde74d402d897a7286086c6199"\nROOT_URL = f"https://raw.githubusercontent.com/amazon-science/spot-diff/{REVISION}/"\nSPLIT_SHA = "a48557e6033318cb90556f706196bc9d247a776a23ea51aecee5a80dd0332995"\nREADME_SHA = "803d1a86c80fcdebc63dfabd041f78336c433ca635b735668aadb24bfc124917"\nARCHIVE_URL = "https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar"\nARCHIVE_BYTES = 1_929_840_640\n\n\ndef fetch_metadata(relative, expected):\n    with urlopen(ROOT_URL + relative, timeout=60) as response:\n        content = response.read(2_000_001)\n    if len(content) > 2_000_000 or hashlib.sha256(content).hexdigest() != expected:\n        raise ValueError("OFFICIAL_METADATA_CHECKSUM_MISMATCH")\n    return content\n\n\ndef normalized_member(name):\n    while name.startswith("./"):\n        name = name[2:]\n    path = PurePosixPath(name)\n    if path.is_absolute() or ".." in path.parts or "\\\\" in name or ":" in name:\n        raise ValueError("UNSAFE_ARCHIVE_MEMBER")\n    if path.parts and path.parts[0] == "VisA":\n        path = PurePosixPath(*path.parts[1:])\n    return path.as_posix()\n\n\ndef audit_archive(archive, split_bytes, output, expected_count=10821):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    split_rows = list(csv.DictReader(io.StringIO(split_bytes.decode("utf-8-sig"))))\n    expected = {row["image"]: row for row in split_rows}\n    if len(split_rows) != len(expected) or len(expected) != expected_count:\n        raise ValueError("UNEXPECTED_OFFICIAL_SPLIT")\n    if any(row["split"] not in {"train", "test"} or (row["split"] == "train" and row["label"] != "normal")\n           for row in split_rows):\n        raise ValueError("NONSTANDARD_TRAIN_TEST_SPLIT")\n    records, seen = [], set()\n    hashes = {"sha256": defaultdict(list), "rgb_sha256": defaultdict(list)}\n    with tarfile.open(archive, "r:*") as tar:\n        for member in tar:\n            name = normalized_member(member.name)\n            if name not in expected:\n                continue\n            if name in seen or not member.isfile() or not 0 < member.size <= 32_000_000:\n                raise ValueError("INVALID_OR_DUPLICATE_IMAGE_MEMBER")\n            seen.add(name)\n            with tar.extractfile(member) as stream:\n                raw = stream.read(32_000_001)\n            if len(raw) != member.size:\n                raise ValueError("IMAGE_SIZE_MISMATCH")\n            with Image.open(io.BytesIO(raw)) as image:\n                rgb = image.convert("RGB")\n                rgb.load()\n                pixels = np.asarray(rgb)\n                thumbnail = np.asarray(rgb.convert("L").resize((9, 8)))\n                bits = (thumbnail[:, 1:] > thumbnail[:, :-1]).flatten()\n                dhash = sum(int(value) << index for index, value in enumerate(bits))\n                shape = list(pixels.shape)\n                decoded_hash = hashlib.sha256(pixels.tobytes()).hexdigest()\n            row = expected[name]\n            record = {"image_path": name, "category": row["object"], "source_split": row["split"],\n                      "role": row["split"], "label": row["label"], "shape": shape,\n                      "sha256": hashlib.sha256(raw).hexdigest(), "rgb_sha256": decoded_hash,\n                      "dhash64": f"{dhash:016x}", "bytes": len(raw),\n                      "physical_instance_id": None}\n            records.append(record)\n            for kind in hashes:\n                hashes[kind][record[kind]].append(record)\n            if len(records) % 1000 == 0:\n                print(json.dumps({"audited_images": len(records), "expected": expected_count}), flush=True)\n    if seen != set(expected):\n        raise ValueError("OFFICIAL_IMAGES_MISSING_FROM_ARCHIVE")\n    duplicate_groups, cross_split = [], []\n    for kind, groups in hashes.items():\n        for digest, items in groups.items():\n            if len(items) > 1:\n                item = {"kind": kind, "hash": digest, "images": [r["image_path"] for r in items],\n                        "roles": sorted({r["role"] for r in items})}\n                duplicate_groups.append(item)\n                if len(item["roles"]) > 1:\n                    cross_split.append(item)\n    manifest = output / "images.jsonl"\n    manifest.write_text("".join(json.dumps(row, sort_keys=True) + "\\n" for row in sorted(records, key=lambda r: r["image_path"])),\n                        encoding="utf-8")\n    (output / "duplicates.json").write_text(json.dumps(duplicate_groups, indent=2), encoding="utf-8")\n    report = {"status": "PIXEL_AUDIT_COMPLETE", "images": len(records),\n              "counts": dict(Counter(r["role"] for r in records)), "categories": dict(Counter(r["category"] for r in records)),\n              "manifest_sha256": hashlib.sha256(manifest.read_bytes()).hexdigest(),\n              "duplicate_groups_by_hash_type": dict(Counter(r["kind"] for r in duplicate_groups)),\n              "cross_split_duplicate_groups": len(cross_split),\n              "split_sha256": hashlib.sha256(split_bytes).hexdigest(), "source_revision": REVISION,\n              "license": "CC BY 4.0 (data, per pinned official README)",\n              "training_ready": False, "test_used_for_training_or_metrics": False,\n              "limits": ["Near-duplicate and cross-dataset review pending.",\n                         "Official split has no verified physical-instance lineage IDs.",\n                         "Industrial anomaly data is not paired parcel handover ground truth.",\n                         "Mask files are not audited or used here; only normal official train is a pretraining candidate."]}\n    (output / "audit.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    return report\n\n\ndef run(output="/kaggle/working/visa-audit", temp="/kaggle/temp/visa-audit"):\n    output, temp = Path(output), Path(temp)\n    output.mkdir(parents=True, exist_ok=True)\n    temp.mkdir(parents=True, exist_ok=True)\n    split = fetch_metadata("split_csv/1cls.csv", SPLIT_SHA)\n    readme = fetch_metadata("README.md", README_SHA)\n    if b"CC BY 4.0" not in readme:\n        raise ValueError("DATA_LICENSE_NOT_FOUND")\n    (output / "1cls.csv").write_bytes(split)\n    (output / "official-README.md").write_bytes(readme)\n    archive = temp / "VisA_20220922.tar"\n    digest, total = hashlib.sha256(), 0\n    with urlopen(ARCHIVE_URL, timeout=60) as response:\n        with archive.open("wb") as stream:\n            for chunk in iter(lambda: response.read(1024 * 1024), b""):\n                total += len(chunk)\n                if total > ARCHIVE_BYTES:\n                    raise ValueError("OFFICIAL_ARCHIVE_SIZE_CHANGED")\n                stream.write(chunk)\n                digest.update(chunk)\n    if total != ARCHIVE_BYTES:\n        raise ValueError("OFFICIAL_ARCHIVE_SIZE_CHANGED")\n    download = {"url": ARCHIVE_URL, "bytes": total, "sha256": digest.hexdigest(),\n                "archive_location": "KAGGLE_TEMP_ONLY_NOT_EXPORTED"}\n    (output / "download.json").write_text(json.dumps(download, indent=2), encoding="utf-8")\n    report = audit_archive(archive, split, output)\n    print(json.dumps(report), flush=True)\n    return report\n', 'ml/data/cross_source_audit.py': '"""Canonical pixel fingerprints and bounded near-duplicate screening, not training.\n\nReads source pixels only for dataset hygiene. No images, thumbnails, labels or\nmodel results from the held-out sets are exported as examples for model tuning.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport heapq\nimport io\nimport json\nimport tarfile\nfrom collections import Counter, defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nfrom urllib.request import urlopen\n\nimport cv2\nimport numpy as np\nfrom PIL import Image\n\nfrom ml.data.parcel3d_audit import find_annotations, within_root\nfrom ml.data.visa_audit import ARCHIVE_BYTES, ARCHIVE_URL, normalized_member\n\nPARCEL_SHA = "267dd46092d3429dc66f539fb897e79b4be3f68beb94b75cd19f7289250e4d38"\nVISA_SHA = "fba1c68957b8e79056696b0a1af96acca1b9faddb0ebc77353819cb56c80e730"\nARCHIVE_SHA = "2eb8690c803ab37de0324772964100169ec8ba1fa3f7e94291c9ca673f40f362"\nBIT_COUNTS = np.array([i.bit_count() for i in range(256)], dtype=np.uint8)\n\n\ndef load_manifest(path, expected_sha):\n    content = Path(path).read_bytes()\n    if hashlib.sha256(content).hexdigest() != expected_sha:\n        raise ValueError("AUDIT_MANIFEST_IDENTITY_CHANGED")\n    rows = [json.loads(line) for line in content.splitlines()]\n    if len({r["image_path"] for r in rows}) != len(rows):\n        raise ValueError("DUPLICATE_MANIFEST_PATH")\n    return rows\n\n\ndef bit_hash(bits):\n    value = sum(int(bit) << index for index, bit in enumerate(np.asarray(bits).flatten()))\n    return f"{value:016x}"\n\n\ndef fingerprint(raw, expected_sha):\n    digest = hashlib.sha256(raw).hexdigest()\n    if digest != expected_sha:\n        raise ValueError("SOURCE_IMAGE_BYTES_CHANGED")\n    with Image.open(io.BytesIO(raw)) as source:\n        rgb = source.convert("RGB")\n        rgb.load()\n        shape = [rgb.height, rgb.width, 3]\n        canonical = hashlib.sha256(f"RGB:{rgb.width}:{rgb.height}:".encode() + rgb.tobytes()).hexdigest()\n        gray = rgb.convert("L")\n        small = np.asarray(gray.resize((9, 8), Image.Resampling.LANCZOS))\n        dhash = bit_hash(small[:, 1:] > small[:, :-1])\n        low = cv2.dct(np.float32(gray.resize((32, 32), Image.Resampling.LANCZOS)))[:8, :8].copy()\n        # Ignore DC in the threshold and set its bit to zero explicitly.\n        bits = low > np.median(low.flatten()[1:])\n        bits[0, 0] = False\n        return {"sha256": digest, "canonical_rgb_sha256": canonical, "shape": shape,\n                "dhash64": dhash, "phash64": bit_hash(bits), "gray_std": float(np.std(small))}\n\n\ndef identify(row, source, raw):\n    return {"dataset": source, "image_path": row["image_path"], "role": row["role"],\n            "lineage_id": row.get("lineage_id"), "category": row.get("category"),\n            **fingerprint(raw, row["sha256"])}\n\n\ndef scope(row):\n    return f"{row[\'dataset\']}:{row[\'role\']}"\n\n\ndef cross_boundary(a, b):\n    return a["dataset"] != b["dataset"] or a["role"] != b["role"]\n\n\ndef exact_groups(rows):\n    output = []\n    for field in ("sha256", "canonical_rgb_sha256"):\n        groups = defaultdict(list)\n        for index, row in enumerate(rows):\n            groups[row[field]].append(index)\n        for digest, indices in groups.items():\n            if len(indices) > 1:\n                output.append({"field": field, "digest": digest, "indices": indices,\n                               "cross_boundary": len({scope(rows[i]) for i in indices}) > 1})\n    return output\n\n\ndef distances(values, query):\n    xor = np.bitwise_xor(values, np.uint64(query))\n    return BIT_COUNTS[xor.view(np.uint8).reshape(-1, 8)].sum(axis=1)\n\n\ndef screen(rows, threshold=4, sample_limit=40):\n    """Count all qualifying boundary pairs; retain deterministic closest examples.\n\n    Both full-frame hashes must be close. This is a screening rule, not a\n    duplicate verdict, and does not detect all crops/rotations or shared meshes.\n    """\n    if not 0 <= threshold <= 64 or sample_limit < 1:\n        raise ValueError("INVALID_DUPLICATE_SCREEN_PARAMETERS")\n    p = np.array([int(r["phash64"], 16) for r in rows], dtype=np.uint64)\n    d = np.array([int(r["dhash64"], 16) for r in rows], dtype=np.uint64)\n    scopes = np.array([scope(r) for r in rows])\n    counts, samples = Counter(), defaultdict(list)\n    comparisons = 0\n    for i in range(len(rows) - 1):\n        indices = np.flatnonzero(scopes[i + 1:] != scopes[i]) + i + 1\n        comparisons += len(indices)\n        pd = distances(p[indices], p[i])\n        chosen = pd <= threshold\n        indices, pd = indices[chosen], pd[chosen]\n        dd = distances(d[indices], d[i])\n        for j, dp, dh in zip(indices[dd <= threshold], pd[dd <= threshold], dd[dd <= threshold], strict=True):\n            j, dp, dh = int(j), int(dp), int(dh)\n            boundary = " <> ".join(sorted((str(scopes[i]), str(scopes[j]))))\n            counts[boundary] += 1\n            heap = samples[boundary]\n            # Negated lexicographic key: largest distance/index is discarded first.\n            item = (-max(dp, dh), -(dp + dh), -i, -j, dp, dh)\n            if len(heap) < sample_limit:\n                heapq.heappush(heap, item)\n            elif item > heap[0]:\n                heapq.heapreplace(heap, item)\n    candidates = []\n    for boundary, heap in sorted(samples.items()):\n        for _, _, ni, nj, dp, dh in sorted(heap, reverse=True):\n            candidates.append({"boundary": boundary, "indices": [-ni, -nj],\n                               "phash_distance": dp, "dhash_distance": dh,\n                               "verdict": "UNREVIEWED_NOT_PROVEN_DUPLICATE"})\n    return {"boundary_comparisons": comparisons, "candidate_pairs": sum(counts.values()),\n            "candidate_counts_by_boundary": dict(sorted(counts.items())),\n            "phash_and_dhash_max_distance": threshold, "retained_per_boundary_limit": sample_limit,\n            "retained_candidates": candidates,\n            "review_status": "PENDING" if candidates else "NO_CANDIDATES_AT_THIS_RULE"}\n\n\ndef fetch_archive(path):\n    digest, total = hashlib.sha256(), 0\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with urlopen(ARCHIVE_URL, timeout=60) as response, path.open("wb") as stream:\n        for chunk in iter(lambda: response.read(1024 * 1024), b""):\n            total += len(chunk)\n            if total > ARCHIVE_BYTES:\n                raise ValueError("OFFICIAL_ARCHIVE_SIZE_CHANGED")\n            digest.update(chunk)\n            stream.write(chunk)\n    if total != ARCHIVE_BYTES or digest.hexdigest() != ARCHIVE_SHA:\n        raise ValueError("OFFICIAL_ARCHIVE_IDENTITY_CHANGED")\n\n\ndef run(input_root, parcel_manifest, visa_manifest, output, temp):\n    cv2.setNumThreads(1)\n    output, temp = Path(output), Path(temp)\n    output.mkdir(parents=True, exist_ok=True)\n    parcel = load_manifest(parcel_manifest, PARCEL_SHA)\n    visa = load_manifest(visa_manifest, VISA_SHA)\n    root = find_annotations(Path(input_root))["train"].parent\n\n    def inspect_parcel(row):\n        return identify(row, "parcel3d", within_root(root, row["image_path"]).read_bytes())\n\n    rows = []\n    with ThreadPoolExecutor(max_workers=4) as pool:\n        for index, result in enumerate(pool.map(inspect_parcel, parcel)):\n            rows.append(result)\n            if (index + 1) % 1000 == 0:\n                print(json.dumps({"dataset": "parcel3d", "decoded": index + 1}), flush=True)\n    archive = temp / "VisA_20220922.tar"\n    fetch_archive(archive)\n    expected, seen = {r["image_path"]: r for r in visa}, set()\n    with tarfile.open(archive) as tar:\n        for member in tar:\n            name = normalized_member(member.name)\n            if name not in expected:\n                continue\n            if name in seen or not member.isfile() or not 0 < member.size <= 32_000_000:\n                raise ValueError("INVALID_OR_DUPLICATE_IMAGE_MEMBER")\n            with tar.extractfile(member) as stream:\n                raw = stream.read(32_000_001)\n            if len(raw) != member.size:\n                raise ValueError("IMAGE_SIZE_MISMATCH")\n            rows.append(identify(expected[name], "visa", raw))\n            seen.add(name)\n            if len(seen) % 1000 == 0:\n                print(json.dumps({"dataset": "visa", "decoded": len(seen)}), flush=True)\n    if seen != set(expected):\n        raise ValueError("OFFICIAL_IMAGES_MISSING_FROM_ARCHIVE")\n    rows.sort(key=lambda r: (r["dataset"], r["image_path"]))\n    manifest = output / "fingerprints.jsonl"\n    manifest.write_text("".join(json.dumps(r, sort_keys=True) + "\\n" for r in rows), encoding="utf-8")\n    exact, near = exact_groups(rows), screen(rows)\n    (output / "exact-groups.json").write_text(json.dumps(exact, indent=2), encoding="utf-8")\n    (output / "near-screen.json").write_text(json.dumps(near, indent=2), encoding="utf-8")\n    report = {"status": "CANONICAL_AUDIT_COMPLETE_REVIEW_REQUIRED", "images": len(rows),\n              "scope_counts": dict(sorted(Counter(scope(r) for r in rows).items())),\n              "manifest_sha256": hashlib.sha256(manifest.read_bytes()).hexdigest(),\n              "input_manifest_hashes": {"parcel3d": PARCEL_SHA, "visa": VISA_SHA},\n              "visa_archive_sha256": ARCHIVE_SHA, "exact_groups": len(exact),\n              "exact_cross_boundary_groups": sum(g["cross_boundary"] for g in exact),\n              "near_candidate_pairs": near["candidate_pairs"], "training_ready": False,\n              "test_pixels_used_for": "HYGIENE_ONLY_NO_TRAINING_OR_MODEL_EVALUATION",\n              "split_reassigned": False, "serving_changed": False,\n              "fingerprint_definition": "RGB:width:height: prefix; LANCZOS right>left dHash; 32px DCT pHash DC=0",\n              "limits": ["Perceptual matches are review candidates, not proof of leakage.",\n                         "Full-frame screening may miss crops, rotations and shared objects with new views.",\n                         "VisA physical-instance identities remain unknown.",\n                         "No training data or existing source manifest is modified.",\n                         "No pixels are exported; source licenses remain unchanged."]}\n    (output / "audit.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    print(json.dumps(report), flush=True)\n    return report\n'}
root=Path('/kaggle/working/_src')
for name,source in SOURCES.items():
 p=root/name
 p.parent.mkdir(parents=True,exist_ok=True)
 p.write_text(source,encoding='utf-8')
sys.path.insert(0,str(root))
Path('/kaggle/working/source-hashes.json').write_text(json.dumps({'ml/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/data/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/data/parcel3d_audit.py': '08ddf5f683f5b23c8364228d53f8b3883518465c3e2d66a64bf3ff9888f46b4e', 'ml/data/visa_audit.py': '7e021357a360737ad93b444b0baa5cbd119139c6de4e5d04843deae5f1180ed5', 'ml/data/cross_source_audit.py': 'dd37a81ee0785f16b90ffcfcbf5aa8688bf6fd53c3bfae089302bc261697b98e'},indent=2))


In [ ]:
from collections import deque
from ml.data.cross_source_audit import PARCEL_SHA,VISA_SHA,run
queue=deque([(Path('/kaggle/input'),0)])
manifests={}
while queue:
 directory,depth=queue.popleft()
 path=directory/'images.jsonl'
 if path.is_file():
  digest=hashlib.sha256(path.read_bytes()).hexdigest()
  if digest in {PARCEL_SHA,VISA_SHA}: manifests[digest]=path
 if depth<6:
  for child in directory.iterdir():
   if child.is_dir() and child.name not in {'parcel3d','train','validation','test','misc'}:
    queue.append((child,depth+1))
assert set(manifests)=={PARCEL_SHA,VISA_SHA},'AUDIT_MANIFEST_MISSING'
run('/kaggle/input',manifests[PARCEL_SHA],manifests[VISA_SHA],
 '/kaggle/working/cross-source-audit','/kaggle/temp/kc-cross-source')
